# Independent full-wave controls
Two historical studies answer different questions:
1. **23.8 GHz planar slabs:** openEMS transmission versus the analytic slab reference,
   with mesh and absorbing-boundary (PML) checks.
2. **60 GHz leaf/wood cross-sections:** fixed supplied first-order edge formulations
   plus a common coherent geometrical-optics (GO) baseline versus openEMS fields.

The second study is **2D, infinite extrusion, Ez polarization and plane-wave incidence
at 10 degrees**. It does not test native path discovery, finite trees or measured
antenna illumination. Agreement in one control does not establish general accuracy.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/replay.py").is_file())
sys.path.insert(0, str(ROOT / "notebooks"))
from replay import verify_data, read_json, dataframe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
%matplotlib inline
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.2, "figure.dpi": 110})
manifest = verify_data()
print(f"Verified {len(manifest['files'])} bundled historical data files.")


In [ ]:
slabs = dataframe("ris_controls_23_8ghz/fullwave_comparison.csv")
mesh = slabs.loc[slabs.pml_cells == 12].copy()
assert len(slabs) == 13 and slabs["passed"].all()
finest = mesh.loc[mesh.groupby("material")["dz_mm"].idxmin()]
display(finest[["material", "thickness_mm", "dz_mm", "transmission_error_db", "phase_error_deg"]])
assert finest["transmission_error_db"].abs().max() < 0.00032
assert finest["phase_error_deg"].abs().max() < 0.0027
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for material, group in mesh.groupby("material"):
    group = group.sort_values("dz_mm", ascending=False)
    axes[0].plot(group.dz_mm, group.transmission_error_db.abs(), "o-", label=material)
    axes[1].plot(group.dz_mm, group.phase_error_deg.abs(), "o-", label=material)
for axis in axes:
    axis.set_xscale("log")
    axis.invert_xaxis()
    axis.set_xlabel("Normal mesh step (mm); finer →")
    axis.legend()
axes[0].set(ylabel="Absolute transmission error (dB)", title="23.8 GHz slab amplitude")
axes[1].set(ylabel="Absolute phase error (degrees)", title="Reference-plane-aligned phase")
plt.show()


The finite-volume slab replaces air. Its empty-normalized transmission includes the
displaced-air phase correction `exp(+j k0 d)`. The supplied table already applies it.
At 23.8 GHz a 0.3 mm slab corresponds to about 8.574 degrees of free-space phase;
adding that correction again would invalidate this comparison. Leaf/wood/pot
material values are stated assumptions, not newly measured plant properties.


In [ ]:
leaf_pml = slabs.loc[(slabs.material == "leaf") & (slabs.dz_mm == 0.0125)].set_index("pml_cells")
display(leaf_pml[["loss_db", "transmission_error_db", "phase_error_deg", "passed"]])
print("12-to-20-cell PML loss change:", abs(leaf_pml.loc[20, "loss_db"] - leaf_pml.loc[12, "loss_db"]), "dB")


## Fixed-edge 60 GHz comparison
Recompute errors from the retained point values and compare with the historical
summaries. The source calls the wood square `branch`; it is a 20 mm square
cross-section, not a finite circular branch. Twelve observation points per object
remain in the calculation, including the worst errors.


In [ ]:
edge = read_json("edge_fullwave_60ghz/analysis.json")
contract = read_json("edge_fullwave_60ghz/contract.json")
gate = read_json("edge_fullwave_60ghz/verification.json")
rows = pd.DataFrame(edge["rows"])
ratio = np.array([complex(*v) for v in rows.fdtd_ratio])
np.testing.assert_allclose(-20 * np.log10(np.abs(ratio)), rows.fdtd_loss_db, atol=1e-10)
summaries = []
for case, group in rows.groupby("case"):
    previous = next(s for s in edge["summaries"] if s["case"] == case)
    for engine in ["sionna", "differt"]:
        errors = group[f"{engine}_loss_db"] - group.fdtd_loss_db
        np.testing.assert_allclose(errors, group[f"{engine}_loss_error_db"], atol=1e-10)
        mae = errors.abs().mean()
        np.testing.assert_allclose(mae, previous[engine]["mean_absolute_loss_error_db"], atol=1e-10)
        summaries.append({"object": case, "formulation": engine, "points": len(group),
                          "mean absolute error (dB)": mae, "worst absolute error (dB)": errors.abs().max()})
errors = pd.DataFrame(summaries)
display(errors)
errors.pivot(index="object", columns="formulation", values="mean absolute error (dB)").plot.bar(
    color=["#137c8b", "#d18738"], rot=0)
plt.ylabel("Mean absolute loss error versus openEMS (dB)")
plt.title("Fixed-edge formulations, 60 GHz, 2D Ez plane wave")
plt.tight_layout()
plt.show()


In [ ]:
CASE = "branch"  # leaf or branch
group = rows.loc[rows.case == CASE]
assert len(group) == 12
fig, axis = plt.subplots(figsize=(10, 4), layout="constrained")
for field, label, color in [("fdtd_loss_db", "openEMS", "#202b40"),
                            ("sionna_loss_db", "Sionna formulation", "#d18738"),
                            ("differt_loss_db", "DiffeRT formulation", "#137c8b")]:
    axis.plot(group["index"], group[field], "o-", label=label, color=color)
axis.set_xticks(group["index"], [f"({x:g}, {y:g})" for x, y in zip(group.x_mm, group.y_mm)], rotation=50)
axis.set(xlabel="Receiver (x, y), mm", ylabel="Coherent excess loss (dB)", title=f"{CASE}: all retained probes")
axis.legend()
plt.show()


## A failed physical limit survives numerical acceptance
Replacing the object material by air should remove its effect: normalized total field
should approach 1. Both retained edge constructions fail that limit. Keep this
result alongside the passing mesh/domain gates. This study does not justify a
general production-model replacement.


In [ ]:
zero = read_json("edge_fullwave_60ghz/zero_contrast.json")
null_rows = []
for case, engines in zero["summary"].items():
    for engine, values in engines.items():
        null_rows.append({"object": case, "formulation": engine, **values})
display(pd.DataFrame(null_rows))
assert gate["numerical_gates_passed"] is True
assert gate["transparent_material_consistency"] is False
print(gate["transparent_material_note"])
print("Historical native kernel check:", gate["native_sionna_matrix_count"], "Sionna components")
display(Markdown("**Model limits:**\n\n" + "\n".join("- " + item for item in contract["limitations"])))
